# 01 — Fetch and Cache Raw Data

StatsBomb: full event data. understat: per-shot xG. fbref (soccerdata): lineups + exact sub timing.
Idempotent — re-running skips already-cached files.

In [ ]:
import sys, json
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))
RAW_DIR = Path('../data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from src.statsbomb_loader import (
    get_available_seasons, get_matches, get_starters,
    get_substitutions, get_shots, get_player_names
)

seasons = get_available_seasons()
print(seasons)

In [ ]:
for _, season in seasons.iterrows():
    season_id = int(season['season_id'])
    season_name = season['season_name']
    out_file = RAW_DIR / f'sb_matches_{season_id}.json'
    if out_file.exists():
        print(f'[SKIP] {season_name}')
        continue

    print(f'[FETCH] {season_name} ...')
    matches = get_matches(season_id)
    records = []

    for _, match in matches.iterrows():
        mid = int(match['match_id'])
        home = str(match['home_team'])
        away = str(match['away_team'])
        try:
            starters = get_starters(mid)
            subs = get_substitutions(mid, home, away)
            shots = get_shots(mid)
            names = get_player_names(mid)
        except Exception as e:
            print(f'  [ERROR] match {mid}: {e}')
            continue

        records.append({
            'match_id': f'sb_{mid}',
            'date': str(match['match_date']),
            'home_team': home,
            'away_team': away,
            'home_team_id': home,
            'away_team_id': away,
            'season': season_name,
            'source': 'statsbomb',
            'starters': starters,
            'substitutions': subs,
            'shots': shots,
            'player_names': names
        })

    with open(out_file, 'w') as f:
        json.dump(records, f)
    print(f'  Saved {len(records)} matches to {out_file.name}')

## understat (shots) + fbref (lineups)

For each season: pre-fetch fbref lineup data once (soccerdata caches to disk),
then fetch shots per match from understat. Lineups come from fbref; shots from understat.
Falls back to understat approximate lineups if fbref has no match for a game.

In [ ]:
from src.understat_loader import get_season_results, get_match_shots, parse_shots
from src.understat_loader import get_match_roster, parse_starters, parse_substitutions
from src.fbref_loader import get_season_lineups, normalize_team_name

UNDERSTAT_YEARS = [2020, 2021, 2022, 2023, 2024]

for year in UNDERSTAT_YEARS:
    season_label = f'{year}-{str(year + 1)[-2:]}'
    out_file = RAW_DIR / f'us_matches_{year}.json'
    if out_file.exists():
        print(f'[SKIP] {season_label}')
        continue

    print(f'[FETCH lineups] {season_label} via fbref ...')
    fbref_lineups = get_season_lineups(year)
    print(f'  fbref returned {len(fbref_lineups)} matches')

    print(f'[FETCH shots]   {season_label} via understat ...')
    results = get_season_results(year)
    records = []
    fbref_hits, fallback_hits = 0, 0

    for match in results:
        mid = int(match['id'])
        home_team = match['h']['title']
        away_team = match['a']['title']
        date_str = match['datetime'][:10]

        try:
            shots_raw = get_match_shots(mid)
        except Exception as e:
            print(f'  [ERROR] shots match {mid}: {e}')
            continue

        home_shots = parse_shots(shots_raw, 'h', home_team, mid)
        away_shots = parse_shots(shots_raw, 'a', away_team, mid)

        match_key = f'{date_str}|{normalize_team_name(home_team)}|{normalize_team_name(away_team)}'
        lineup = fbref_lineups.get(match_key)

        if lineup:
            home_starters = lineup['home_starters']
            away_starters = lineup['away_starters']
            subs = lineup['substitutions']
            player_names = lineup['player_names']
            fbref_hits += 1
        else:
            try:
                roster_raw = get_match_roster(mid)
            except Exception as e:
                print(f'  [WARN] roster fallback failed match {mid}: {e}')
                continue
            home_starters = parse_starters(roster_raw, 'h')
            away_starters = parse_starters(roster_raw, 'a')
            subs = (
                parse_substitutions(roster_raw, 'h', home_team)
                + parse_substitutions(roster_raw, 'a', away_team)
            )
            player_names = {
                f"us_{p['id']}": p['player']
                for p in roster_raw.get('h', []) + roster_raw.get('a', [])
            }
            fallback_hits += 1

        records.append({
            'match_id': f'us_{mid}',
            'date': date_str,
            'home_team': home_team,
            'away_team': away_team,
            'home_team_id': home_team,
            'away_team_id': away_team,
            'season': season_label,
            'source': 'understat',
            'starters': {home_team: home_starters, away_team: away_starters},
            'substitutions': subs,
            'shots': home_shots + away_shots,
            'player_names': player_names
        })

    with open(out_file, 'w') as f:
        json.dump(records, f)
    print(f'  Saved {len(records)} matches (fbref: {fbref_hits}, fallback: {fallback_hits})')

print('Done!')